# Test Stage 3 — Does the frequency shifting work, and in the right direction?

**What this stage checks:** the mixer. That the signal really is being slid down to zero
frequency, and — just as important — that a signal *above* the reference tone can be told
apart from one *below* it.

That second point sounds like a detail. It is not. If the sign convention is wrong, every
spectrum you ever record will be **mirrored left to right**, and it will look completely
plausible. The ion dip would appear on the wrong side of the resonator and nothing would
obviously be amiss. It is worth ten minutes to rule out.

---

## What you need

| | |
|---|---|
| Stage 2 | All four checks passing |
| Signal generator | Sine wave into **IN1**, amplitude a few hundred mV |
| Frequency | Set `F_SIG` below to match it. 1 MHz is a good choice |

## The idea

The mixer multiplies your input by an internal reference tone. Multiplication produces a
component at the **difference** of the two frequencies, so:

| Reference tone set to | Difference | What we should see |
|---|---|---|
| exactly `F_SIG` | 0 Hz | Both outputs sit still |
| `F_SIG` − 1 kHz | +1 kHz | Outputs rotate, one way |
| `F_SIG` + 1 kHz | −1 kHz | Outputs rotate, the **other** way |

"Rotate" is the right word. The two outputs, I and Q, are the horizontal and vertical
coordinates of a point going round a circle. When the input and reference match exactly the
point stops moving. When they differ by 1 kHz it goes round once per millisecond — and which
way round tells you which of the two frequencies is higher.

## How we watch it

The scope tap sends the mixer's I output to the scope's channel A and Q to channel B, so
both are captured simultaneously. Combining them as a single complex number lets us measure
a **signed** frequency: positive means the input is above the reference tone.

## Setup

Run both cells. Then set `F_SIG` in the next section to your generator's frequency.

In [ ]:
import os, mmap, struct, time
import numpy as np

FS_CLK = 125e6          # FPGA processing clock [Hz]

# =============================================================================
#  Acquisition system driver  -  register block at 0x40600000 (sys[6])
# =============================================================================
#  Full register map is in the header of red_pitaya_acq.v and in
#  acquisition_custom/README.md.
#
#  CRITICAL RULE: every access is a SINGLE 32-bit read or write. A bulk mmap
#  slice becomes a burst transfer that the GP-AXI to system-bus bridge cannot
#  acknowledge; it locks the bus and reboots the board. Do not "optimise" the
#  loops below into slice operations.
# =============================================================================
ACQ_BASE = 0x40600000

R_CTRL       = 0x00   # [0] enable  [1] soft reset  [2] test mode  [3] scope tap
R_LO_FINC    = 0x04   # local-oscillator frequency word
R_DEC_LOG2   = 0x08   # log2 of the decimation factor, 6..12
R_SRC_SEL    = 0x0C   # 0 = IN1, 1 = IN2
R_TAP_SEL    = 0x10   # 0 = mixer output, 1 = CIC output
R_TAP_SHIFT  = 0x14   # which 14-bit window of the tapped value to show
R_PEEK_I     = 0x18   # most recent I sample (read-only)
R_PEEK_Q     = 0x1C   # most recent Q sample (read-only)
R_SAMPLE_CNT = 0x20   # output samples since reset (read-only)
R_STATUS     = 0x24   # [0] enabled [1] test mode [2] tap active (read-only)


def s32(v):
    """Reinterpret a 32-bit register value as a signed number.

    Registers come back as unsigned integers, but I and Q are signed. Anything
    with the top bit set is negative, and the value is 2**32 lower than it looks.
    """
    return v - (1 << 32) if v & 0x80000000 else v


class Acq(object):
    def __init__(self, base=ACQ_BASE):
        self.fd = os.open('/dev/mem', os.O_RDWR | os.O_SYNC)
        self.m  = mmap.mmap(self.fd, 0x1000, offset=base)

    def w(self, off, val):
        self.m[off:off+4] = struct.pack('<I', val & 0xFFFFFFFF)

    def r(self, off):
        return struct.unpack('<I', self.m[off:off+4])[0]

    # ---- frequency word -----------------------------------------------------
    @staticmethod
    def finc(f_lo):
        """Convert a wanted local-oscillator frequency [Hz] into the register value.

        The oscillator is a counter that represents an angle; its full range is
        one complete turn. Adding `finc` every clock tick advances the angle, so

            f_out = f_clk * finc / 2**32     =>     finc = f_out / f_clk * 2**32

        Step size is 125e6 / 2**32 = 0.0291 Hz.
        """
        return int(round(f_lo / FS_CLK * (1 << 32))) & 0xFFFFFFFF

    @staticmethod
    def finc_to_hz(finc):
        """Inverse of finc(): the frequency the hardware will actually produce."""
        return finc * FS_CLK / (1 << 32)

    # ---- configuration ------------------------------------------------------
    def setup(self, f_lo, dec_log2, src=0, test=False,
              tap_en=False, tap_sel=0, tap_shift=10):
        """Configure and start the chain.

        THE ORDER MATTERS. The decimator holds a running total that assumes one
        fixed decimation factor; changing it while data is flowing leaves a
        meaningless partial sum in the pipeline and produces a burst of nonsense.
        So the sequence is always: disable, write settings, reset, enable.
        """
        self.w(R_CTRL, 0)                       # 1. disable
        self.w(R_LO_FINC,   self.finc(f_lo))    # 2. settings
        self.w(R_DEC_LOG2,  dec_log2)
        self.w(R_SRC_SEL,   src)
        self.w(R_TAP_SEL,   tap_sel)
        self.w(R_TAP_SHIFT, tap_shift)
        ctrl = 1 | (4 if test else 0) | (8 if tap_en else 0)
        self.w(R_CTRL, ctrl | 2)                # 3. enable + one-shot reset
        self.w(R_CTRL, ctrl)                    # 4. running

    def disable(self):
        self.w(R_CTRL, 0)

    def reset(self):
        """Pulse the soft reset, leaving the other control bits as they are."""
        ctrl = self.r(R_CTRL) & 0xD             # keep enable / test / tap bits
        self.w(R_CTRL, ctrl | 2)
        self.w(R_CTRL, ctrl)

    # ---- readback -----------------------------------------------------------
    def peek(self):
        """Most recent (I, Q) output sample, as signed integers."""
        return s32(self.r(R_PEEK_I)), s32(self.r(R_PEEK_Q))

    def count(self):
        return self.r(R_SAMPLE_CNT)


acq = Acq()
print('acquisition system mapped at 0x%08X' % ACQ_BASE)
print('STATUS = 0x%08X' % acq.r(R_STATUS))

In [ ]:
# =============================================================================
#  Internal scope driver  -  register block at 0x40100000 (sys[1])
# =============================================================================
#  Taken unchanged from phaseshift_custom/scope_phase_check.ipynb, which is
#  already proven on this board.
#
#  NOTE: the equalisation filter this used to configure has been REMOVED from
#  the design (it caused 1173 timing violations and was switched off anyway -
#  see CHANGELOG.md). The coefficient registers still exist and still accept
#  writes, they simply have no effect. For these measurements that is an
#  improvement: the scope is now a flat, unmodified path.
# =============================================================================
SC_BASE = 0x40100000
SC_MAP  = 0x40000
BUF_A   = 0x10000
BUF_B   = 0x20000
NMAX    = 1 << 14        # 16384 samples per channel


class Scope(object):
    def __init__(self):
        self.fd = os.open('/dev/mem', os.O_RDWR | os.O_SYNC)
        self.m  = mmap.mmap(self.fd, SC_MAP, offset=SC_BASE)

    def w(self, off, val):
        self.m[off:off+4] = struct.pack('<I', val & 0xFFFFFFFF)

    def r(self, off):
        return struct.unpack('<I', self.m[off:off+4])[0]

    def _buf(self, off, n):
        # One 32-bit word at a time - see the warning in the acquisition driver.
        m  = self.m
        ba = bytearray(4 * n)
        for i in range(n):
            o = off + 4 * i
            ba[4*i:4*i+4] = m[o:o+4]
        v = np.frombuffer(bytes(ba), dtype='<u4').astype(np.int64) & 0x3FFF
        # The buffer holds 14-bit two's-complement values. Anything with bit 13
        # set is negative and must have 0x4000 subtracted.
        return np.where(v & 0x2000, v - 0x4000, v).astype(float)

    def acquire(self, dec=1, n=NMAX):
        self.w(0x14, dec)       # scope's own decimation (1 = every sample)
        self.w(0x10, n - 1)     # how many samples to record
        self.w(0x00, 2)         # reset the acquisition engine
        self.w(0x00, 1)         # arm
        self.w(0x04, 1)         # trigger now, from software
        t0 = time.time()
        while self.r(0x04) != 0:            # clears itself when the capture ends
            if time.time() - t0 > 1.0:
                break
        time.sleep(0.005)
        return self._buf(BUF_A, n), self._buf(BUF_B, n), FS_CLK / dec


sc = Scope()
print('internal scope mapped at 0x%08X' % SC_BASE)

In [ ]:
# ---------------------------------------------------------------------------
#  Remove the circular-buffer wrap from captures
#
#  The scope writes into a ring buffer and keeps writing until the capture ends,
#  so the oldest sample is wherever the write pointer stopped - NOT at index 0.
#  Reading from the start of the buffer therefore returns a rotated copy of the
#  time series, with one discontinuity in the middle. That is the single step
#  visible in the Step 2 traces.
#
#  Register 0x18 holds the current write pointer. Rolling the arrays by it puts
#  the samples back in time order.
#
#  Run this cell once, after the Scope class has been created. It replaces the
#  acquire method on the existing object.
# ---------------------------------------------------------------------------
def _acquire_unwrapped(self, dec=1, n=NMAX):
    self.w(0x14, dec)       # scope decimation
    self.w(0x10, n - 1)     # samples to record
    self.w(0x00, 2)         # reset the acquisition engine
    self.w(0x00, 1)         # arm
    self.w(0x04, 1)         # trigger now, from software
    t0 = time.time()
    while self.r(0x04) != 0:
        if time.time() - t0 > 1.0:
            break
    time.sleep(0.005)

    a = self._buf(BUF_A, n)
    b = self._buf(BUF_B, n)

    if n == NMAX:                       # only meaningful for a full-buffer capture
        wp = self.r(0x18) & (NMAX - 1)  # where the writing stopped
        a = np.roll(a, -wp)             # oldest sample first
        b = np.roll(b, -wp)

    return a, b, FS_CLK / dec


import types
sc.acquire = types.MethodType(_acquire_unwrapped, sc)
print('scope captures will now be un-wrapped (write pointer read from 0x18)')
print('the single step in the middle of the traces should disappear')

---
## Step 1 — Find the signal, and set the tap range

First look at the raw input with the tap **off**, so the scope sees IN1 directly. This
confirms the generator is connected and tells us its exact frequency and amplitude.

**Passes if:** the measured frequency matches your generator and the amplitude is a few
thousand counts. Full scale is 8191, so aim for 2000–6000 — enough to be well above the
noise, with room to spare before clipping.

In [ ]:
# ===========================================================================
#  THE ONE LINE YOU MUST EDIT
#
#  F_SIG is what your signal generator is set to. The measurement window is only
#  a few tens of kilohertz wide out of a 62 MHz input range, so a wrong value
#  here does not make the signal weak - it makes it VANISH, exactly as if the
#  cable were unplugged. Getting this wrong cost two full runs on 2026-07-29.
# ===========================================================================
F_SIG = 10.0e6          # <-- YOUR SIGNAL GENERATOR FREQUENCY [Hz]


def est_freq(x, fs):
    """Dominant frequency of a real signal, refined between FFT bins."""
    x = x - x.mean()
    X = np.abs(np.fft.rfft(x * np.hanning(len(x))))
    k = int(np.argmax(X[1:])) + 1
    if 1 <= k < len(X) - 1:
        l, c, r = (np.log(X[k-1] + 1e-9), np.log(X[k] + 1e-9), np.log(X[k+1] + 1e-9))
        k = k + 0.5 * (l - r) / (l - 2*c + r)
    return k * fs / len(x)


acq.disable()                      # tap off -> the scope sees IN1 as usual
time.sleep(0.02)
a, b, fs = sc.acquire(dec=1)

f_in   = est_freq(a, fs)
amp_in = np.abs(a).max()

print('measured input frequency : %.1f Hz   (you set F_SIG = %.1f Hz)' % (f_in, F_SIG))
print('measured input amplitude : %.0f counts   (full scale is 8191)' % amp_in)
print()

if amp_in < 500:
    print('** input is very small - turn the generator up, or check the cable **')
elif amp_in > 7800:
    print('** input is clipping - turn the generator down **')
else:
    print('input level is fine')

# ALWAYS adopt the MEASURED frequency from here on, rather than the number you
# typed. A generator set to "10 MHz" is typically a few tens of hertz off, and
# every later step wants the reference tone sitting on the signal as exactly as
# possible. (A 37 Hz error is harmless here - it makes the I-Q point rotate once
# every 27 ms, against a capture lasting 0.13 ms - but there is no reason to
# carry it.)
if abs(f_in - F_SIG) / F_SIG > 0.05:
    print('** measured frequency is a long way from what you typed - check the generator **')
F_SIG = f_in
print('F_SIG now set to the measured %.1f Hz' % F_SIG)

---
## Step 2 — Set the reference tone equal to the signal

With the reference exactly on the signal, the difference frequency is zero, so the wanted
part of the mixer output should **stop moving** and sit at constant values.

### But the raw mixer output is never flat — and that is correct

Multiplying two tones produces **two** new components: one at the difference of the
frequencies, and one at their **sum**. Both come out of the mixer at exactly the same size.

With a 10 MHz signal and a 10 MHz reference, that means the tap carries:

| Component | Frequency | Amplitude |
|---|---|---|
| wanted, the difference | ~0 Hz | A × 256 |
| unwanted, the sum | **20 MHz** | A × 256 — *the same* |

So a raw capture looks like a **solid band**, not a line: a fast 20 MHz oscillation of full
amplitude sitting on top of the slowly-varying value we actually want. At 125 million samples
per second, 20 MHz is only 6.25 samples per cycle, which fills the plot in.

**This is the mixer working properly.** Removing the sum term is the decimator's job, and the
decimator is the next block along — it is not in this measurement. (The same effect is noted
in the mixer's source header and in `validate_dds_mixer.py`, where searching the whole
spectrum finds the sum term instead of the signal.)

So before judging flatness we apply a **moving average in software**, which does the same job
the decimator does in hardware: average over more than one cycle of the 20 MHz term and it
cancels, leaving the baseband value.

**Passes if:** the *smoothed* traces are flat — the variation is small compared with the
distance of the point from the origin.

**If it fails:** if the smoothed traces still oscillate at a steady frequency, the reference
tone is not where you think it is. Read that frequency off the plot; it is the error between
your generator and the reference.

In [ ]:
def autorange_tap(start=14):
    """Choose the 14-bit window of the tapped value that best fits the scope's range."""
    best = start
    for shift in range(start, -1, -1):
        acq.w(R_TAP_SHIFT, shift)
        time.sleep(0.01)
        a, b, _ = sc.acquire(dec=1, n=4096)
        peak = max(np.abs(a).max(), np.abs(b).max())
        if peak > 7500:
            break
        best = shift
        if peak > 2500:
            break
    acq.w(R_TAP_SHIFT, best)
    return best


# Reference tone exactly on the signal; tap shows the MIXER output (TAP_SEL = 0).
acq.setup(f_lo=F_SIG, dec_log2=10, tap_en=True, tap_sel=0, tap_shift=14)
time.sleep(0.05)
shift = autorange_tap()
a, b, fs = sc.acquire(dec=1)

SMOOTH = 64          # moving-average length, in samples


def lowpass(x, n=SMOOTH):
    """Moving average - the software equivalent of what the decimator does.

    The raw mixer output carries the unwanted sum term at twice the signal
    frequency, at full amplitude. Averaging over more than one of its cycles
    cancels it and leaves the baseband value we actually want to look at.

    At 10 MHz the sum term is at 20 MHz, which is 6.25 samples per cycle, so 64
    samples covers about ten cycles - ample. 'valid' mode trims the partial
    windows at each end rather than letting them distort the result.
    """
    return np.convolve(x, np.ones(n) / n, mode='valid')


a_s, b_s = lowpass(a), lowpass(b)
mag_s = np.sqrt(a_s**2 + b_s**2)

print('TAP_SHIFT : %d' % shift)
print()
print('RAW (contains the 20 MHz sum term - expected to be a wide band):')
print('  I : mean %+8.1f   spread %7.1f' % (a.mean(), a.std()))
print('  Q : mean %+8.1f   spread %7.1f' % (b.mean(), b.std()))
print()
print('SMOOTHED (sum term averaged away - this is what should be flat):')
print('  I : mean %+8.1f   spread %7.1f' % (a_s.mean(), a_s.std()))
print('  Q : mean %+8.1f   spread %7.1f' % (b_s.mean(), b_s.std()))
print('  magnitude sqrt(I^2+Q^2) : mean %8.1f   spread %6.1f' % (mag_s.mean(), mag_s.std()))
print()

# "Flat" means the variation is small compared with how far the point sits from
# the origin. Comparing to the magnitude rather than to I or Q separately avoids
# a false alarm when one of them happens to sit near zero.
flatness = max(a_s.std(), b_s.std()) / max(mag_s.mean(), 1.0)
print('smoothed variation / magnitude = %.3f   (%.1f dBc)'
      % (flatness, 20*np.log10(max(flatness, 1e-9))))
print('STEP 2:', 'PASS' if flatness < 0.3 else 'FAIL - outputs are still moving')
if flatness > 0.02:
    print()
    print('   NOTE: a residual of this size means something is arriving at the input')
    print('   OFFSET from your reference tone, as well as the wanted signal. The mixer')
    print('   cannot create that - it is a multiplier, so every output component maps')
    print('   one-to-one onto an input component. Run the diagnostic cell below to')
    print('   find out what and where it is. It does not invalidate this step.')
print()
print('For reference, the raw band should be about %.0f counts wide -' % (2 * a.std() * 1.41))
print('that is the 20 MHz sum term, and it is supposed to be there.')

In [ ]:
import matplotlib.pyplot as plt

n_show = 4000
t_raw = np.arange(n_show) / fs * 1e3                       # milliseconds
t_sm  = np.arange(min(n_show, len(a_s))) / fs * 1e3

fig, ax = plt.subplots(1, 3, figsize=(15, 4))

# --- raw: expected to be a solid band, that is the 20 MHz sum term ---
ax[0].plot(t_raw, a[:n_show], lw=0.4, label='I raw')
ax[0].plot(t_raw, b[:n_show], lw=0.4, label='Q raw')
ax[0].set_xlabel('time [ms]'); ax[0].set_ylabel('tapped value')
ax[0].set_title('RAW mixer output.\nThe band IS the 20 MHz sum term - expected')
ax[0].legend(loc='upper right', fontsize=8); ax[0].grid(True)

# --- smoothed: this is what should be flat ---
ax[1].plot(t_sm, a_s[:len(t_sm)], label='I smoothed')
ax[1].plot(t_sm, b_s[:len(t_sm)], label='Q smoothed')
ax[1].set_xlabel('time [ms]'); ax[1].set_ylabel('tapped value')
ax[1].set_title('SMOOTHED (sum term averaged away):\nboth should be flat')
ax[1].legend(loc='upper right', fontsize=8); ax[1].grid(True)

# --- the same smoothed data as a path in the I-Q plane ---
n_iq = min(len(a_s), 8000)
ax[2].plot(a_s[:n_iq], b_s[:n_iq], '.', ms=1)
ax[2].set_xlabel('I'); ax[2].set_ylabel('Q')
ax[2].set_title('I-Q plane, smoothed:\nshould be a blob, not a ring')
ax[2].axis('equal'); ax[2].grid(True)

plt.tight_layout(); plt.show()

print('If the LEFT panel is a solid band, that is correct - it is the sum term at')
print('twice the signal frequency, which the decimator removes further down the chain.')
print('Only the MIDDLE and RIGHT panels are evidence about whether the frequency')
print('shift landed on zero.')

### Diagnostic — identifying anything left over

If the smoothed traces are not quite flat, or the I-Q plot is a ring centred away from the
origin rather than a blob, then the output is **a constant plus a rotating term**. A rotating
term means a component offset in frequency from your reference tone.

**The mixer cannot have created it.** A multiplier is linear in its input: every component
coming out corresponds to exactly one component going in, shifted by the reference frequency.
So if something orbits at 1 MHz in the output, something was sitting 1 MHz from the reference
at the input.

This cell finds it and then looks for it directly at the input, which settles where it comes
from. The usual answer is a spur from the signal generator — cheap function generators often
have one a few hundred kilohertz to a few megahertz from the carrier.

**It does not necessarily matter.** The decimator has a clean band of only a few kilohertz at
useful settings, so anything a megahertz away is removed entirely and never reaches your data.
The point of identifying it is to know it comes from outside the instrument.

In [ ]:
# ---- 1. what is left over in the mixer output? ----
z  = (a_s - a_s.mean()) + 1j * (b_s - b_s.mean())     # smoothed, carrier removed
Z  = np.abs(np.fft.fft(z * np.hanning(len(z))))
fz = np.fft.fftfreq(len(z), d=1.0 / fs)
k  = np.argmax(Z)
f_residual = fz[k]

carrier = np.hypot(a_s.mean(), b_s.mean())

print('MIXER OUTPUT (smoothed)')
print('  carrier (the constant part)  : %8.1f counts' % carrier)
print('  largest rotating component   : %+9.1f Hz' % f_residual)
print('  its size relative to carrier : %8.1f %%  (%.1f dBc)'
      % (100*a_s.std()*1.414/carrier, 20*np.log10(a_s.std()*1.414/carrier)))
print()
print('  -> implies a component at %.4f MHz at the INPUT' % ((F_SIG + f_residual)/1e6))
print()

# ---- 2. is it there at the input? ----
acq.disable()                       # tap off, so the scope sees IN1 directly
time.sleep(0.05)
raw, _, fsr = sc.acquire(dec=1)

R  = np.abs(np.fft.rfft((raw - raw.mean()) * np.hanning(len(raw))))
fr = np.fft.rfftfreq(len(raw), d=1.0 / fsr)
R  = R / R.max()                    # normalise to the carrier

print('INPUT SPECTRUM on IN1 - everything within 1 dB..40 dB below the carrier:')
print('  %12s %10s' % ('freq [MHz]', 'dBc'))
peaks = []
for idx in range(2, len(R) - 2):
    if R[idx] > R[idx-1] and R[idx] > R[idx+1] and R[idx] > 10**(-40/20.0):
        peaks.append((fr[idx], 20*np.log10(R[idx])))
peaks.sort(key=lambda t: -t[1])
for f, d in peaks[:8]:
    tag = '   <-- the carrier' if d > -1 else ''
    print('  %12.4f %10.1f%s' % (f/1e6, d, tag))

print()
print('If a peak appears near %.3f MHz, the residual comes from the signal'
      % ((F_SIG + f_residual)/1e6))
print('generator or the cabling, not from anything in the FPGA.')

plt.figure(figsize=(10, 4))
plt.plot(fr/1e6, 20*np.log10(R + 1e-12), lw=0.7)
plt.axvline(F_SIG/1e6, color='g', ls='--', lw=1, label='reference tone')
plt.axvline((F_SIG + f_residual)/1e6, color='r', ls='--', lw=1, label='predicted residual')
plt.xlim(0, 30); plt.ylim(-90, 5)
plt.xlabel('frequency [MHz]'); plt.ylabel('dBc')
plt.title('What is actually arriving on IN1')
plt.legend(); plt.grid(True, alpha=0.4); plt.tight_layout(); plt.show()

---
## Step 3 — The important one: offset the reference and check the direction

Move the reference tone away from the signal by a known amount, first one way then the other,
and measure the resulting rotation.

We combine the two outputs into a single complex number, `z = I + jQ`, and take its Fourier
transform. Unlike a single real signal, a complex signal has a **two-sided** spectrum — it can
distinguish +30 kHz from −30 kHz. That is the entire reason the mixer has two multipliers.

### How small an offset can this measurement resolve?

There is one rule, and it has no escape clause:

$$\text{bin spacing} \;=\; \frac{1}{\text{total acquisition time}}$$

It depends only on **how long you watch for**, never on how fast you sample. Sampling faster
buys bandwidth; only watching longer buys resolution.

The scope captures 16384 samples at 125 MSPS — 131 microseconds — so the bins are
**7629 Hz** apart. Two frequencies closer together than that land in the same bin and cannot
be told apart at all.

This caught out the first version of this test, which used offsets of 5 and 10 kHz. Those are
0.66 and 1.31 bins, so both landed in bin 1 and both read 7629 Hz. The *signs* were right —
which is what this step is actually for — but the magnitudes were meaningless.

`DELTA` below is therefore chosen automatically as a multiple of the bin spacing, and the peak
is refined by interpolating between neighbouring bins, which typically recovers a tenth of a
bin.

**Expected, from the sign convention built into the hardware:**

| Reference tone | Input relative to reference | Measured peak |
|---|---|---|
| `F_SIG − Δ` | Δ above | **+Δ** |
| `F_SIG + Δ` | Δ below | **−Δ** |

**Passes if:** every measured frequency matches the expected value *including its sign*.

**If the magnitudes are right but every sign is inverted:** the minus on the quadrature
product has been lost. Every spectrum from this instrument would come out mirrored — the ion
dip on the wrong side of the resonator, looking entirely plausible. Fix it in
`red_pitaya_acq_mixer.v` before going any further.

In [ ]:
# If Step 2 has not been run in this kernel, `shift` will not exist. Fall back to
# a sensible default rather than failing with a NameError.
try:
    shift
except NameError:
    shift = 8
    acq.w(R_TAP_SHIFT, shift)
    print('NOTE: `shift` was undefined (Step 2 not run) - defaulting to %d' % shift)

# ---------------------------------------------------------------------------
#  Resolution of this measurement
#
#  bin spacing = 1 / acquisition time. The scope gives 16384 samples at
#  125 MSPS = 131 us, so bins are 7629 Hz apart. Offsets must be several bins
#  wide to be measurable, so DELTA is derived from the bin spacing rather than
#  picked by hand.
# ---------------------------------------------------------------------------
NFFT  = 16384
BIN   = fs / NFFT              # frequency resolution of one capture
DELTA = 4 * BIN                # ~30 kHz: four bins, comfortably resolvable

print('capture duration : %.1f us' % (NFFT / fs * 1e6))
print('bin spacing      : %.1f Hz   (= 1 / capture duration)' % BIN)
print('DELTA chosen     : %.1f Hz   (%.1f bins)' % (DELTA, DELTA / BIN))
print()


def measure_offset(f_lo, remove_dc=True):
    """Set the reference tone, capture I and Q, return the SIGNED baseband frequency."""
    acq.setup(f_lo=f_lo, dec_log2=10, tap_en=True, tap_sel=0, tap_shift=shift)
    time.sleep(0.05)
    a, b, fs_l = sc.acquire(dec=1)

    # Combine into one complex signal. This is what makes a signed frequency
    # measurable: a real signal cannot tell +f from -f, a complex one can.
    z = a + 1j * b
    if remove_dc:
        # Strip the stationary part so the ROTATING component is what we find.
        # (Do not do this when the expected offset is zero - it would remove the
        #  very thing being measured, which is what made the zero case report a
        #  meaningless 7629 Hz in the first version of this test.)
        z = z - z.mean()

    Z     = np.fft.fft(z * np.hanning(len(z)))
    freqs = np.fft.fftfreq(len(z), d=1.0 / fs_l)
    mag   = np.abs(Z)

    # Look only near baseband. The mixer output also carries a large component
    # at twice the signal frequency; it is real and expected, and the decimator
    # is what removes it. Searching the whole spectrum would find that instead.
    near = np.abs(freqs) < 200e3
    idx  = np.where(near)[0]
    k    = idx[np.argmax(mag[idx])]

    # Refine the peak by fitting a parabola through the log of the three bins
    # around it. A pure tone rarely sits exactly on a bin centre; without this
    # the answer is quantised to 7629 Hz, which was the original failure.
    n = len(z)
    l  = np.log(mag[(k - 1) % n] + 1e-12)
    c  = np.log(mag[k]           + 1e-12)
    r  = np.log(mag[(k + 1) % n] + 1e-12)
    den = l - 2 * c + r
    frac = 0.5 * (l - r) / den if den != 0 else 0.0
    frac = np.clip(frac, -0.5, 0.5)

    f = ((k + frac) % n) * fs_l / n
    if f > fs_l / 2:
        f -= fs_l                       # wrap into the signed range
    return f, mag[k]


# The zero-offset case is checked separately: with no rotation there is no peak
# to find, so the right question is whether the energy sits at DC instead.
acq.setup(f_lo=F_SIG, dec_log2=10, tap_en=True, tap_sel=0, tap_shift=shift)
time.sleep(0.05)
a0, b0, _ = sc.acquire(dec=1)
z0 = a0 + 1j * b0
dc_fraction = np.abs(z0.mean()) / np.sqrt(np.mean(np.abs(z0) ** 2))
print('reference exactly on the signal:')
print('   fraction of the energy sitting at 0 Hz = %.3f  (want > 0.5)' % dc_fraction)
print('   %s' % ('OK' if dc_fraction > 0.5 else 'CHECK - see Step 2'))
print()

cases = [
    (F_SIG - 1*DELTA, +1*DELTA, 'signal is ABOVE the reference'),
    (F_SIG + 1*DELTA, -1*DELTA, 'signal is BELOW the reference'),
    (F_SIG - 2*DELTA, +2*DELTA, 'signal is further above'),
    (F_SIG + 2*DELTA, -2*DELTA, 'signal is further below'),
    (F_SIG - 4*DELTA, +4*DELTA, 'signal well above'),
    (F_SIG + 4*DELTA, -4*DELTA, 'signal well below'),
]

print('%14s %14s %14s %11s %9s  %s'
      % ('reference [Hz]', 'expected [Hz]', 'measured [Hz]', 'error [Hz]', 'in bins', 'result'))
print('-' * 88)
all_ok = True
for f_lo, expect, label in cases:
    meas, _ = measure_offset(f_lo)
    err = meas - expect
    # With interpolation a tenth of a bin is realistic; allow a quarter.
    tol = max(0.25 * BIN, abs(expect) * 0.02)
    ok  = abs(err) < tol
    all_ok &= ok
    print('%14.0f %+14.0f %+14.0f %+11.0f %9.2f  %s'
          % (f_lo, expect, meas, err, err / BIN, 'OK' if ok else 'FAIL'))
    print('               %s' % label)

print()
print('STEP 3:', 'PASS' if all_ok else 'FAIL')
if not all_ok:
    print('   Check the SIGNS first. If every sign matches and only the sizes are out,')
    print('   this is a resolution problem, not a physics one - increase DELTA.')
    print('   If the sizes are right but every sign is inverted, the minus sign on')
    print('   the quadrature product in red_pitaya_acq_mixer.v has been lost.')

### Watch it rotate

With the reference offset from the signal, the point traces a **circle** in the I-Q plane
instead of sitting still. The direction it goes round is the sign of the frequency offset —
this plot is the sign convention made visible.

In [ ]:
# ---------------------------------------------------------------------------
#  Direction-of-rotation plot
#
#  FIX: this used to plot three full rotations, so the dark-to-light colour
#  gradient wrapped three times and every point on the circle ended up with
#  several colours over it - the direction of travel, which is the entire point
#  of these panels, was invisible.
#
#  Now it plots THREE QUARTERS of one rotation, with a marker at the start and
#  one at the end. Less than a full turn means the gradient cannot wrap, so the
#  direction is unambiguous.
# ---------------------------------------------------------------------------
fig, ax = plt.subplots(1, 3, figsize=(14, 4.6))

panels = [(F_SIG,         'reference ON the signal\n(should be a blob)'),
          (F_SIG - DELTA, 'signal ABOVE reference\n(expect ANTICLOCKWISE)'),
          (F_SIG + DELTA, 'signal BELOW reference\n(expect CLOCKWISE)')]

for k, (f_lo, ttl) in enumerate(panels):
    acq.setup(f_lo=f_lo, dec_log2=10, tap_en=True, tap_sel=0, tap_shift=shift)
    time.sleep(0.05)
    a_r, b_r, fs_r = sc.acquire(dec=1)

    # Smooth first. Without this the sum term, comparable in size to the signal,
    # smears every point and hides the circle.
    ai, bi = lowpass(a_r), lowpass(b_r)

    if k == 0:
        n = min(len(ai), 8000)          # no rotation to show, so just show plenty
    else:
        # three quarters of ONE rotation - enough to see the direction, not
        # enough for the colour gradient to wrap
        n = int(0.75 * fs_r / DELTA)
        n = max(200, min(n, len(ai)))

    sc_plot = ax[k].scatter(ai[:n], bi[:n], c=np.arange(n), s=4, cmap='viridis')
    if k > 0:
        ax[k].plot(ai[0], bi[0], 'o', ms=11, mfc='none', mec='k', mew=2)
        ax[k].plot(ai[n-1], bi[n-1], 'x', ms=11, mec='r', mew=2)
        ax[k].annotate('', xy=(ai[n-1], bi[n-1]), xytext=(ai[n-20], bi[n-20]),
                       arrowprops=dict(arrowstyle='-|>', color='r', lw=2))
    ax[k].set_title(ttl); ax[k].set_xlabel('I'); ax[k].set_ylabel('Q')
    ax[k].axhline(0, color='0.8', lw=0.8); ax[k].axvline(0, color='0.8', lw=0.8)
    ax[k].axis('equal'); ax[k].grid(True, alpha=0.4)

plt.suptitle('Open circle = start,  red cross and arrow = end.  '
             'The two right-hand panels must turn OPPOSITE ways.')
plt.tight_layout(); plt.show()

print('Reading these: pick any point on the circle and follow the colour from')
print('dark to light. The middle and right panels must go opposite ways round.')
print('If they turn the SAME way, the sign convention is broken and every')
print('spectrum this instrument produces would come out mirrored.')

---
## Step 4 — Does the amplitude stay put?

As the reference tone moves away from the signal, the *size* of the rotating point should not
change much. The mixer's job is to shift frequency, not to alter amplitude.

Some fall-off at larger offsets is expected and harmless here — it comes from the decimator,
not the mixer — but over a few kilohertz it should be small.

**Passes if:** the magnitude varies by less than about 10% across the offsets tested.

In [ ]:
offsets = np.array([-20e3, -10e3, -5e3, 0, 5e3, 10e3, 20e3])
mags = []

for off in offsets:
    acq.setup(f_lo=F_SIG - off, dec_log2=10, tap_en=True, tap_sel=0, tap_shift=shift)
    time.sleep(0.05)
    a, b, _ = sc.acquire(dec=1, n=8192)
    # Smooth away the sum term first, then measure how far the point sits from
    # the origin. Measuring on the raw data would mostly measure the sum term.
    ai, bi = lowpass(a), lowpass(b)
    mags.append(np.sqrt(ai**2 + bi**2).mean())

mags  = np.array(mags)
spread = (mags.max() - mags.min()) / mags.mean()

print('%12s %14s' % ('offset [kHz]', 'magnitude'))
for o, m in zip(offsets / 1e3, mags):
    print('%12.1f %14.1f' % (o, m))
print()
print('variation across offsets : %.1f %%' % (100 * spread))
print('STEP 4:', 'PASS' if spread < 0.10 else 'CHECK - larger than expected')

plt.figure(figsize=(7, 4))
plt.plot(offsets / 1e3, mags, 'o-')
plt.xlabel('signal offset from reference tone [kHz]')
plt.ylabel('mixer output magnitude')
plt.title('Amplitude should be flat - the mixer shifts frequency, not size')
plt.grid(True); plt.tight_layout(); plt.show()

---
## Stage 3 summary

| Step | What it proves |
|---|---|
| 1 | The generator is connected and at a sensible level |
| 2 | With the reference on the signal, the output stops moving — the shift lands on zero |
| 3 | Offsets appear at the right frequency **and the right sign** |
| 4 | The mixer changes frequency without changing amplitude |

**All four passing** means downconversion works and the spectrum will come out the right way
round. Step 3 is the one that matters most: it is the only check that would catch a mirrored
spectrum, and a mirrored spectrum looks entirely believable.

**Before moving on**, switch the tap off:

In [ ]:
acq.disable()
print('acquisition disabled, scope restored to its normal inputs')